In [1]:
from bs4 import BeautifulSoup as bs

In [2]:
import requests

In [3]:
import re
import json

In [4]:
import pandas as pd

In [5]:
import threading

In [6]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
import time

# def getHtml(url):
#     print(url)
#     options = Options()
#     options.add_argument('--headless')
#     driver = webdriver.Chrome(options=options)
    
#     driver.get(url)
#     time.sleep(3)
#     html = driver.page_source
        
#     driver.quit()
#     return html


    
# def getHtml(url):
#     print(url)
#     options = Options()
#     # options.add_argument('--headless')
#     driver = webdriver.Chrome(options=options)
    
#     driver.get(url)
#     time.sleep(5)
#     html = driver.page_source
#     while("Вы не робот?" in html):
#         print()
#         print()
#         print()
#         print()
#         print(url)
#         print(html)
#         driver.quit()
#         driver = webdriver.Chrome(options=options)
#         driver.get(url)
#         time.sleep(3)
#         html = driver.page_source
    
#     driver.quit()
#     return html




def getHtml(url):
    options = Options()
    options.add_argument('--headless')
    options.add_argument('--no-sandbox')
    options.add_argument('--disable-blink-features=AutomationControlled')
    options.add_argument('--user-agent=Mozilla/5.0 (Windows NT 10.0; win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome 120.0.0.0 Safari/537.36')
    options.add_argument('--disable-dev-shm-usage')
    driver = webdriver.Chrome(options=options)
    driver.execute_script("Object.defineProperty(navigator, 'webdriver', {get: () => undefined})")
    driver.get(url)
    time.sleep(3)
    html = driver.page_source
    while("Вы не робот?" in html):
        print()
        print()
        print()
        print()
        print(url)
        print(html)
        driver.quit()
        driver = webdriver.Chrome(options=options)
        driver.get(url)
        time.sleep(3)
        html = driver.page_source
    driver.quit()
    return html

    
# import undetected_chromedriver as uc
# def getHtml(url):
#     driver = uc.Chrome(headless=True, use_subprocess=False)
#     driver.get(url)
#     time.sleep(3)
#     html = driver.page_source
#     driver.quit()
#     return html

In [7]:
result_list = {'title': [], 'year': [], 'country': [], 'genre': [], 'description': []}

In [8]:
thread_results = [None] * 5

def task(links, page_index):
    local_result = {'title': [], 'year': [], 'country': [], 'genre': [], 'description': []}
    for link in links:
        url = 'https://www.kinopoisk.ru' + str(link.get('href'))
        
        html = getHtml(url)
        soup = bs(html, 'html.parser')
        
        
        titleDiv = soup.find('h1', class_='styles_title__h1USJ styles_root__J7Vae styles_root__krgW7 styles_rootInLight__8xmQ4')
        if(titleDiv == None):
            titleDivDark = soup.find('h1', class_='styles_title__h1USJ styles_root__J7Vae styles_root__krgW7 styles_rootInDark__lhkly')
            try:
                title = titleDivDark.span.text
            except:
                print(soup)
                continue
            title = titleDivDark.span.text
            local_result['title'].append(title)
            
            description = soup.find('p', class_='styles_root__5ffO_').text
            local_result['description'].append(description)
            
            year = soup.find('a', class_='styles_linkLight__GLx8D styles_link__gw947').text
            local_result['year'].append(year)
            
            headers = soup.find_all('div', class_='styles_rowLight__vntr5 styles_row__ehb6N')
            
            for header in headers:
                
                nameOfHeader = header.find('div', 'styles_titleLight__X7Bwk styles_title__hofDs').text
                
                countriesStr = ''
                if(nameOfHeader == 'Страна'):
                    values = header.find_all('a', 'styles_linkLight__GLx8D styles_link__gw947')
                    for value in values:
                        countriesStr += value.text + ' '
                    local_result['country'].append(countriesStr)
            
                genresStr = ''
                if(nameOfHeader == 'Жанр'):
                    values = header.find_all('a', 'styles_linkLight__GLx8D styles_link__gw947')
                    for value in values:
                        genresStr += value.text + ' '
                    local_result['genre'].append(genresStr)
        
        else:
            title = titleDiv.span.text
            local_result['title'].append(title)
            
            description = soup.find('p', class_='styles_paragraph__V0fA2').text
            local_result['description'].append(description)
            
            year = soup.find('a', class_='styles_linkDark__cT_iW styles_link__gw947').text
            local_result['year'].append(year)
            
            headers = soup.find_all('div', class_='styles_rowDark__Q3Dh2 styles_row__ehb6N')
            
            for header in headers:
                
                nameOfHeader = header.find('div', 'styles_titleDark__ghZ_A styles_title__hofDs').text
                
                countriesStr = ''
                if(nameOfHeader == 'Страна'):
                    values = header.find_all('a', 'styles_linkDark__cT_iW styles_link__gw947')
                    for value in values:
                        countriesStr += value.text + ' '
                    local_result['country'].append(countriesStr)
            
                genresStr = ''
                if(nameOfHeader == 'Жанр'):
                    values = header.find_all('a', 'styles_linkDark__cT_iW styles_link__gw947')
                    for value in values:
                        genresStr += value.text + ' '
                    local_result['genre'].append(genresStr)
    thread_results[page_index - 1] = local_result

threads = []
for ind in range (1,6):
    urlPage = 'https://www.kinopoisk.ru/lists/movies/top250/?utm_referrer=organic.kinopoisk.ru&page=' + str(ind)
    html = getHtml(urlPage)
    soup = bs(html, 'html.parser')
    links = soup.find_all('a', class_='styles_poster__u9xhS styles_root__vaZRT')
    
    t = threading.Thread(target=task, args=(links, ind))
    threads.append(t)
    t.start()

for t in threads:
    t.join()

for res in thread_results:
    for key in result_list:
        result_list[key].extend(res[key])







https://www.kinopoisk.ru/film/258687/
<html prefix="og: http://ogp.me/ns#" lang="ru"><head><script src="https://yastatic.net/s3/gdpr/v3/gdpr.js" type="text/javascript" charset="utf-8" async=""></script><meta http-equiv="X-UA-Compatible" content="IE=edge"><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1"><title>Вы не робот?</title><link rel="stylesheet" href="/captcha_smart.c1f6a7cf8d410e04e643.min.css?k=1770297661809"><style>@media only screen and (min-width:651px) and (prefers-color-scheme:light){body{background-image:url('https://captcha-backgrounds.s3.yandex.net/static/kinopoisk-background.jpg')}}@media only screen and (min-width:651px) and (prefers-color-scheme:dark){body{background-image:url('https://captcha-backgrounds.s3.yandex.net/static/kinopoisk-background.jpg')}}@media (prefers-color-scheme:light){.LogoLink{background-image:url('https://cdnrhkgfkkpupuotntfj.svc.cdn.yandex.net/kinopoisklogo.svg')}.Theme_root_default{--smart-captcha-b

In [9]:
result_list

{'title': ['Интерстеллар (2014)',
  'Побег из Шоушенка (1994)',
  'Джентльмены (2019)',
  '1+1 (2011)',
  'Остров проклятых (2009)',
  'Зеленая миля (1999)',
  'Терминатор 2: Судный день (1991)',
  'Бойцовский клуб (1999)',
  'Форрест Гамп (1994)',
  'Властелин колец: Возвращение короля (2003)',
  'Зеленая книга (2018)',
  'Начало (2010)',
  'Унесённые призраками (2001)',
  'Волк с Уолл-стрит (2013)',
  'Властелин колец: Братство кольца (2001)',
  'Криминальное чтиво (1994)',
  'Темный рыцарь (2008)',
  'Брат (1997)',
  'Властелин колец: Две крепости (2002)',
  'Унесённые ветром (1939)',
  'Гладиатор (2000)',
  'Брат 2 (2000)',
  'Леон (1994)',
  'Пятый элемент (1997)',
  'Собачье сердце (ТВ, 1988)',
  'Крестный отец (1972)',
  'Список Шиндлера (1993)',
  'Деревья на асфальте (1984)',
  'В августе 44-го (2001)',
  'Достучаться до небес (1997)',
  'Назад в будущее (1985)',
  'Шрэк (2001)',
  'Тайна Коко (2017)',
  'Операция «Ы» и другие приключения Шурика (1965)',
  'Поймай меня, если с

In [10]:
print("Количество нулевых значений в: ")
for i in result_list:
    print( i + " - " + str(result_list[i].count(None)))

Количество нулевых значений в: 
title - 0
year - 0
country - 0
genre - 0
description - 0


In [11]:
file_name = 'kinoPoisk250top.csv'
df = pd.DataFrame(data=result_list)
df.to_csv(file_name)

In [12]:
df.head()

,title,year,country,genre,description
0,Интерстеллар (2014),2014,США Великобритания Канада,фантастика драма приключения,"Когда засуха, пыльные бури и вымирание растени..."
1,Побег из Шоушенка (1994),1994,США,драма,Бухгалтер Энди Дюфрейн обвинён в убийстве собс...
2,Джентльмены (2019),2019,США Великобритания,криминал комедия боевик,Один ушлый американец ещё со студенческих лет ...
3,1+1 (2011),2011,Франция,драма комедия,"Пострадав в результате несчастного случая, бог..."
4,Остров проклятых (2009),2009,США Канада,триллер детектив драма,Два американских судебных пристава отправляютс...


In [13]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 250 entries, 0 to 249
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   title        250 non-null    object
 1   year         250 non-null    object
 2   country      250 non-null    object
 3   genre        250 non-null    object
 4   description  250 non-null    object
dtypes: object(5)
memory usage: 9.9+ KB
